# Florida Speech 1998 — Q&A Dataset Generation
Source: BuffettUofFloridaspeech.pdf (129 KB)
Primary labels: Psychology, Risk Management, Personal Life, Strategy Development

In [1]:
import sys, re, asyncio
from pathlib import Path

sys.path.insert(0, "..")
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)

### Chunking functions

`_sub_chunk` handles oversized Buffett responses. It splits at paragraph boundaries with overlap so no argument gets cut mid-thought. The overlap means adjacent sub-chunks share some text, which can produce similar Q&A pairs — this is expected and cleaned up by dedup in the assembly notebook.

`chunk_florida_speech` does three things:
1. **Strips page headers** — every page repeats a title and page number. These are PDF artifacts, not content.
2. **Splits on `Question:` delimiters** — the speech has a clean Q&A structure. The opening monologue is extracted separately since it has no question prefix.
3. **Merges runts** — sub-chunking sometimes leaves a trailing fragment too small to generate quality pairs from, so it gets merged back into the previous chunk.

In [2]:
def _sub_chunk(text, source_file, section, max_chars=4000, overlap=500):
    if len(text) <= max_chars:
        return [Chunk(text=text, source_file=source_file,
                      source_section=section, chunk_strategy="qa_exchange")]
    paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    sub_chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current_len + len(para) > max_chars and current:
            sub_chunks.append(Chunk(
                text='\n'.join(current), source_file=source_file,
                source_section=section, chunk_strategy="qa_exchange_subchunk"))
            overlap_paras, overlap_len = [], 0
            for p in reversed(current):
                if overlap_len + len(p) > overlap: break
                overlap_paras.insert(0, p)
                overlap_len += len(p)
            current = overlap_paras + [para]
            current_len = overlap_len + len(para)
        else:
            current.append(para)
            current_len += len(para)
    if current:
        sub_chunks.append(Chunk(
            text='\n'.join(current), source_file=source_file,
            source_section=section,
            chunk_strategy="qa_exchange_subchunk" if sub_chunks else "qa_exchange"))
    return sub_chunks

def chunk_florida_speech(pdf_path):
    raw = extract_text(pdf_path)
    cleaned = re.sub(r'Buffett Talk to MBA Students at Florida University 1998\.\s*\n\d+\s*\n?', '\n', raw)
    parts = re.split(r'\n(?=Questions?:)', cleaned)
    chunks = []
    intro = parts[0].strip()
    if len(intro) > 200:
        bs = intro.find("Buffett: (holds mike)")
        if bs > 0:
            sermon = intro[bs:].strip()
            if len(sermon) > 100:
                chunks.extend(_sub_chunk(sermon, "florida_speech.pdf", "Opening Monologue"))
    for part in parts[1:]:
        part = part.strip()
        if len(part) < 50: continue
        nl = part.find('\n')
        qt = part[:nl].strip() if nl > 0 else part[:80]
        section = re.sub(r'^Questions?:\s*', '', qt)[:80]
        chunks.extend(_sub_chunk(part, "florida_speech.pdf", section))
    merged = []
    for chunk in chunks:
        if len(chunk.text) < 1000 and merged:
            merged[-1].text += '\n' + chunk.text
        else:
            merged.append(chunk)
    return merged

### Run chunking and inspect
Verify the splits look right before spending API calls on classification. Each chunk should have a recognizable topic and enough text to generate from.

In [3]:
chunks = chunk_florida_speech(Path("../sources/BuffettUofFloridaspeech.pdf"))
print(f"Total chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c.source_section[:60]} | {len(c.text)} chars")

Total chunks: 27
  [0] Opening Monologue | 4962 chars
  [1] What about Japan?  Your thoughts about Japan? | 2686 chars
  [2] You were rumored to be one of the rescue buyers of Long Term | 4032 chars
  [3] You were rumored to be one of the rescue buyers of Long Term | 4043 chars
  [4] You were rumored to be one of the rescue buyers of Long Term | 1027 chars
  [5] What makes a company something that you like? | 4004 chars
  [6] What makes a company something that you like? | 4614 chars
  [7] You covered half of it which is trying to understand a busin | 4022 chars
  [8] You covered half of it which is trying to understand a busin | 3100 chars
  [9] If I have every bought a company where the numbers told me n | 4863 chars
  [10] The Asian Crisis and how it affects a company like Coke that | 4002 chars
  [11] The Asian Crisis and how it affects a company like Coke that | 1601 chars
  [12] What about your business mistakes? | 4585 chars
  [13] The current tenuous economic situation and inte

## Q&A Generation
Each chunk gets sent to DeepSeek with a label-specific prompt that enforces:
- Grounding in the source passage (no hallucinated facts)
- 3-5 sentence self-contained answers
- Coverage across sublabels within each label

In [4]:
classified = await classify_chunks(chunks)
save_checkpoint(classified, "florida_classified")

Classifying 27 chunks (skipping 0 pre-labeled)
  10/27
  20/27
  27/27

Label distribution:
  Strategy Development: 16
  Psychology: 5
  Risk Management: 3
  Personal Life: 2
  Timing: 1
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\florida_classified.pkl


### Inspect classifications
Check that labels make sense for each chunk's topic. The Long-Term Capital Management (hedge fund collapse) chunks should land on Risk Management or Psychology. The moat and Coca-Cola chunks should land on Strategy Development. The opening monologue about character traits should land on Psychology or Personal Life. Any unclassified chunks get skipped during generation.

In [5]:
for c in classified:
    print(f"  {c.label:25s} ({c.confidence:.2f}) | {c.source_section[:50]}")
failed = [c for c in classified if c.label is None]
if failed:
    print(f"\nWARNING: {len(failed)} chunks unclassified")

  Psychology                (0.90) | Opening Monologue
  Strategy Development      (0.90) | What about Japan?  Your thoughts about Japan?
  Risk Management           (0.80) | You were rumored to be one of the rescue buyers of
  Risk Management           (0.95) | You were rumored to be one of the rescue buyers of
  Psychology                (0.85) | You were rumored to be one of the rescue buyers of
  Strategy Development      (0.95) | What makes a company something that you like?
  Strategy Development      (0.95) | What makes a company something that you like?
  Strategy Development      (0.95) | You covered half of it which is trying to understa
  Strategy Development      (0.90) | You covered half of it which is trying to understa
  Strategy Development      (0.90) | If I have every bought a company where the numbers
  Strategy Development      (0.90) | The Asian Crisis and how it affects a company like
  Timing                    (0.90) | The Asian Crisis and how it affects a compa

### Generate Q&A pairs
Each chunk gets 3 candidate pairs via DeepSeek. The prompt is label-specific — a Risk Management chunk gets a prompt focused on loss prevention and position sizing, while a Psychology chunk gets one focused on temperament and discipline. Checkpoint saves raw pairs so you can resume from here if scoring fails later.

In [6]:
pairs = await generate_all(classified, n_pairs=3)
save_checkpoint(pairs, "florida_pairs_raw")
print(f"\nTotal raw pairs: {len(pairs)}")

  5/27 chunks | 15 pairs
  10/27 chunks | 30 pairs
  15/27 chunks | 45 pairs
  20/27 chunks | 60 pairs
  25/27 chunks | 75 pairs
  27/27 chunks | 81 pairs
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\florida_pairs_raw.pkl

Total raw pairs: 81


In [7]:
#a few samples
for p in pairs[:5]:
    print(f"[{p.label}] Q: {p.question}")
    print(f"         A: {p.answer[:150]}...")
    print()

[Psychology] Q: According to Warren Buffett in this passage, what personal qualities are more important for success than intellect and energy?
         A: Buffett argues that qualities of behavior, temperament, and character are more critical for success than sheer intellect and energy. He illustrates th...

[Psychology] Q: How does Buffett use the concept of "going short" on a classmate to highlight negative psychological traits?
         A: Buffett introduces a "hooker" by asking which classmate you would choose to "go short" on, meaning you would bet against their success. He states you ...

[Psychology] Q: What does Buffett suggest is the key difference between the qualities we admire and those we dislike in others?
         A: Buffett suggests that the qualities we admire on the "left hand side"—like integrity, leadership, generosity, and honesty—are behavioral and temperame...

[Strategy Development] Q: How does Warren Buffett's 'cigar butt' investing approach differ from his pre

### Score and filter pairs
Each pair gets scored against its source chunk by GPT-4o-mini on groundedness, label fit, richness, and novelty. The chunk map links each pair back to the chunk it was generated from so the scorer can verify the answer actually follows from the source text. Anything below 0.7 composite score gets dropped.

In [3]:
chunk_map = {c.chunk_id: c for c in classified}
scored = await score_all(pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "florida_pairs_filtered")

  Scored 10/81
  Scored 20/81
  Scored 30/81
  Scored 40/81
  Scored 50/81
  Scored 60/81
  Scored 70/81
  Scored 80/81
  Scored 81/81
Quality filter: 67/81 passed (threshold=0.7)
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\florida_pairs_filtered.pkl


### Score distribution
If the mean is above 0.75, generation quality is solid. Below 0.70 means the generation prompt needs tuning. A tight range (e.g. 0.75–0.90) means consistent quality. A wide range means some chunks produced much better pairs than others.

In [4]:
import statistics
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Pairs after filtering: {len(filtered)}")
print(f"Score range: {min(scores):.2f} - {max(scores):.2f}")
print(f"Mean: {statistics.mean(scores):.2f} | Median: {statistics.median(scores):.2f}")

Pairs after filtering: 67
Score range: 0.73 - 0.91
Mean: 0.84 | Median: 0.86


In [2]:
print("Coverage for Florida speech:")
coverage_audit(filtered)
export_csv(filtered, Path("../intermediate/florida_qa.csv"))
export_detailed(filtered, Path("../intermediate/florida_qa_detailed.csv"))

Coverage for Florida speech:
  Personal Life               5 pairs ( 7.5%)
    Sublabels hit: early_life, habits, personal_values
  Strategy Development       38 pairs (56.7%)
    Sublabels hit: value_investing_framework, margin_of_safety, competitive_moat, business_quality, circle_of_competence, capital_allocation
  Timing                      3 pairs ( 4.5%)
    Sublabels hit: entry_criteria, patience, market_cycles
  Risk Management             9 pairs (13.4%)
    Sublabels hit: position_sizing, diversification, leverage_avoidance, permanent_loss, margin_of_safety_risk, concentration
  Adaptability                0 pairs ( 0.0%)
  Psychology                 12 pairs (17.9%)
    Sublabels hit: temperament, emotional_discipline, contrarian_thinking, patience_psychology, fear_greed, rationality

  Total: 67 pairs
Exported 67 pairs to ..\intermediate\florida_qa.csv
  Strategy Development: 38
  Psychology: 12
  Risk Management: 9
  Personal Life: 5
  Timing: 3
Detailed export: ..\interme

In [1]:
#reload checkpoints and restart in case of reprocessing
# Idempotent reload — run this cell anytime after a kernel restart
import sys, statistics, importlib
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)

# Load latest checkpoints
classified = load_checkpoint("florida_classified")
pairs = load_checkpoint("florida_pairs_raw")
filtered = load_checkpoint("florida_pairs_filtered")
print(f"Loaded: {len(classified)} chunks | {len(pairs)} raw pairs | {len(filtered)} filtered pairs")

Loaded: 27 chunks | 81 raw pairs | 67 filtered pairs
